In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LlamaGQA(nn.Module):
    def __init__(self, dim, num_query_heads, num_kv_heads, head_dim=None):
        super().__init__()
        self.dim = dim
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        ## 注意这里是除以query_heads，一个query对应多个kv
        self.head_dim = head_dim if head_dim is not None else dim // num_query_heads
        # 每个查询头映射到特定KV头
        self.kv_groups = num_query_heads // num_kv_heads
        assert num_query_heads % num_kv_heads == 0, "num_query_heads必须被num_kv_heads整除"
        # 投影矩阵
        self.q_proj = nn.Linear(dim, self.num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_query_heads * self.head_dim, dim, bias=False)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, attention_mask=None, cache=None):
        batch_size, seq_len, _ = x.shape
        # 计算查询、键、值
        q = self.q_proj(x).view(batch_size, seq_len, self.num_query_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        # 变换维度以便计算注意力
        q = q.transpose(1, 2)  # [batch_size, num_query_heads, seq_len, head_dim]
        k = k.transpose(1, 2)  # [batch_size, num_kv_heads, seq_len, head_dim]
        v = v.transpose(1, 2)  # [batch_size, num_kv_heads, seq_len, head_dim]
        # 使用KV缓存
        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
            cache = (k, v)
        # 实现分组注意力：重复k和v以匹配查询头数
        if self.num_query_heads > self.num_kv_heads:
            k = k.repeat_interleave(self.kv_groups, dim=1)
            v = v.repeat_interleave(self.kv_groups, dim=1)
        # 计算注意力分数
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        # 应用注意力掩码
        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask
        # 应用softmax获取注意力权重
        attn_weights = F.softmax(attn_scores, dim=-1)
        # 计算输出
        output = torch.matmul(attn_weights, v)  # [batch_size, num_query_heads, seq_len, head_dim]
        # 重塑输出并进行最终投影
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        output = self.o_proj(output)
        if cache is not None:
            return output, cache
        return output
    

class MLAAttention(nn.Module):
    """
    完整版 MLA (Multi-Head Latent Attention)
    面试核心考点：KV 低秩压缩 + RoPE 专用通路分离
    """
    def __init__(
        self,
        hidden_size: int = 4096,    # 模型隐层维度
        n_heads: int = 32,          # 注意力头数
        latent_kv_dim: int = 512,   # KV 压缩后的低维维度
        rope_theta: float = 10000.0
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_heads = n_heads
        self.head_dim = hidden_size // n_heads  # 每个头的维度
        self.latent_kv_dim = latent_kv_dim      # KV 低秩维度

        # ==================== MLA 核心层 ====================
        # 1. Q 投影：拆成 语义通路(Qc) + RoPE 专用通路(Qr)
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # 2. KV 共享压缩投影：高维 → 低维 latent 空间
        self.kv_latent_proj = nn.Linear(hidden_size, latent_kv_dim * 2, bias=False)
        
        # 3. 低维 → 高维 上采样（推理可融合优化）
        self.k_up_proj = nn.Linear(latent_kv_dim, hidden_size, bias=False)
        self.v_up_proj = nn.Linear(latent_kv_dim, hidden_size, bias=False)
        
        # 4. 输出投影
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        # RoPE
        self.rope_theta = rope_theta

    def precompute_rope_cos_sin(
        max_seq_len: int,  # 最大序列长度（预计算范围）
        head_dim: int,     # 每个注意力头的维度（必须是偶数！RoPE要求）
        device: torch.device,
        rope_theta: float = 10000.0  # RoPE超参，大模型默认10000
    ):
        """
        预计算RoPE的cos/sin矩阵（推理可缓存，避免重复计算）
        输出shape: [1, max_seq_len, 1, head_dim] → 适配Q/K的[B, H, T, D]广播
        """
        # 1. 计算频率：theta = 1 / (rope_theta ^ (2i / head_dim)) ，i为特征维度索引
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        # 2. 生成位置索引：[0,1,2,...,max_seq_len-1]
        pos = torch.arange(0, max_seq_len, device=device).float()
        # 3. 频率与位置相乘：[max_seq_len, head_dim/2]
        freqs = torch.outer(pos, inv_freq)
        # 4. 扩展为与head_dim一致的维度（偶数位cos，奇数位sin）
        cos = torch.cos(freqs).repeat_interleave(2, dim=-1)
        sin = torch.sin(freqs).repeat_interleave(2, dim=-1)
        # 增加batch和head维度，方便广播：[1, max_seq_len, 1, head_dim]
        cos = cos.unsqueeze(0).unsqueeze(2)
        sin = sin.unsqueeze(0).unsqueeze(2)
        return cos, sin

    def apply_rotary_pos_emb(
        q: torch.Tensor,  # Q: [B, H, T, D] (batch, heads, seq_len, head_dim)
        k: torch.Tensor,  # K: [B, H, T_k, D]
        cos: torch.Tensor,# 预计算的cos: [1, max_seq_len, 1, D]
        sin: torch.Tensor # 预计算的sin: [1, max_seq_len, 1, D]
    ):
        # 1. 把最后一维 拆成 偶数位 + 奇数位
        # q = [x0, y0, x1, y1, x2, y2...]
        xq = q[..., 0::2]  # 偶数位：0,2,4...
        yq = q[..., 1::2]  # 奇数位：1,3,5...

        xk = k[..., 0::2]
        yk = k[..., 1::2]

        # cos, sin 也要对应拆分
        xc = cos[..., 0::2]
        xs = sin[..., 0::2]

        # 2. 直接套用 RoPE 公式（最朴素）
        xq_out = xq * xc - yq * xs
        yq_out = xq * xs + yq * xc

        xk_out = xk * xc - yk * xs
        yk_out = xk * xs + yk * xc

        # 3. 把偶数位、奇数位 堆叠回去 → 恢复原形状
        q_out = torch.stack([xq_out, yq_out], dim=-1).flatten(-2)
        k_out = torch.stack([xk_out, yk_out], dim=-1).flatten(-2)

        return q_out, k_out

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor = None,
        sin: torch.Tensor = None,
        past_kv: tuple = None
    ):
        B, T, C = x.shape  # (batch, seq_len, hidden_size)
        H = self.n_heads
        D = self.head_dim

        # ==================== 1. Q 计算 + RoPE ====================
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)  # [B, H, T, D]
        q, _ = apply_rotary_pos_emb(q, q, cos[:, :q.size(2)], sin[:, :q.size(2)])  # Q 走 RoPE

        # ==================== 2. KV 低秩压缩 ====================
        kv_latent = self.kv_latent_proj(x)  # [B, T, 2*latent_dim]
        k_c, v_c = kv_latent.chunk(2, dim=-1)  # 低维 K/V [B, T, latent_dim]

        # KV 缓存
        if past_kv is not None:
            past_k_c, past_v_c = past_kv
            k_c = torch.cat([past_k_c, k_c], dim=1)
            v_c = torch.cat([past_v_c, v_c], dim=1)
        current_kv = (k_c, v_c)

        # ==================== 3. 低维上采样回高维 ====================
        k = self.k_up_proj(k_c).view(B, -1, H, D).transpose(1, 2)  # [B, H, T, D]
        v = self.v_up_proj(v_c).view(B, -1, H, D).transpose(1, 2)

        # K 走 RoPE
        _, k = apply_rotary_pos_emb(q, k, cos[:, :k.size(2)], sin[:, :k.size(2)])

        # ==================== 4. 标准注意力计算 ====================
        scale = D ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = attn @ v

        # ==================== 5. 输出拼接 ====================
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.o_proj(out)

        return out, current_kv
